# 01 — Niche construction & characterisation

EcoFoundation analyses spatial transcriptomics data through **niches** —
local neighbourhoods of cells in tissue. This notebook covers Step 2 + 2.5:

1. Load the AnnData
2. Build niches with each of the four strategies (kNN, Delaunay, radius, tiling)
3. Compare niche-size distributions
4. Compute standardised per-niche characterisation
5. Render the report-style plots interactively

## Why niches?

Single-cell methods describe each cell in isolation; spatial methods need a
scale at which biological context emerges. A niche bundles a cell with its
immediate microenvironment — the GNN downstream treats each niche as one graph.

In [ ]:
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('agg')  # in-notebook display still works

# Ensure the repo root is on the path (works whether kernel is project or system)
REPO = Path.cwd().parent.parent
if str(REPO / 'src') not in sys.path:
    sys.path.insert(0, str(REPO / 'src'))

from ecofoundation.config.schemas import DataConfig, NicheConfig
from ecofoundation.io.readers import load_anndata, validate_schema
from ecofoundation.niches.assembly import assign_niches
from ecofoundation.niches.characterization import compute_niche_stats

## 1. Load the AnnData

We use the demo dataset shipped at the repo root. `DataConfig` is a
Pydantic schema that holds all column-name mappings — change `path` and
the relevant column names for your own data.

In [ ]:
data_cfg = DataConfig(
    path=REPO / 'scVI_adata_annotated.h5ad',
    sample_id_col='samples',
    patient_id_col='patient',
    condition_col='sonicated',
    celltype_col='celltype_level_1',
    spatial_key='spatial',
    counts_layer='counts',
    normalized_layer='X_exp',
    embedding_key='X_scVI',
)
adata = load_anndata(data_cfg)
schema = validate_schema(adata, data_cfg)
schema

## 2. Niche construction — four strategies

`NicheConfig.strategy` controls which strategy is used. EcoFoundation
supports:

- `knn` (default) — fixed-size niches; every ego cell has its 50 nearest spatial neighbours.
- `delaunay` — density-aware k-hop niches on the Delaunay triangulation.
- `radius` — all cells within a µm radius.
- `tiling` — Voronoi partition seeded via Farthest-Point-Sampling (strictly disjoint).

Always patient-aware: a niche cannot span two patients. The overlap controller
caps pairwise Jaccard overlap (default 0.2) for supervised setups; for the
unsupervised pipeline we disable it so every cell gets its own niche.

In [ ]:
knn_niches, _ = assign_niches(
    adata, data_cfg,
    NicheConfig(strategy='knn', knn_k=50, min_cells_per_niche=10,
                overlap_filter_enabled=True, max_overlap_fraction=0.2),
)
print('kNN-50 niches:', knn_niches.n_niches,
      '| median size:', int(np.median(knn_niches.sizes())))

In [ ]:
delaunay_niches, _ = assign_niches(
    adata, data_cfg,
    NicheConfig(strategy='delaunay', k_hop=3,
                edge_length_quantile_cutoff=0.95,
                min_cells_per_niche=8, max_cells_per_niche=300,
                overlap_filter_enabled=True, max_overlap_fraction=0.2),
)
print('Delaunay-3hop niches:', delaunay_niches.n_niches,
      '| size range:', int(delaunay_niches.sizes().min()),
      '..', int(delaunay_niches.sizes().max()))

## 3. Niche characterisation

For every niche we compute:

- `size`, `radius`, `mean_nn_distance` (cellular density proxy)
- `shannon_entropy` of cell-type composition
- `center_purity` (fraction of niche cells matching the ego's cell type)
- `n_unique_celltypes`
- `co_occurrence` matrix (center cell-type × neighbour cell-type)

In [ ]:
stats = compute_niche_stats(adata, knn_niches, data_cfg)
stats.summary()

In [ ]:
stats.per_niche.describe().round(2)

## 4. Visualisations

Every plot in EcoFoundation returns a matplotlib Figure styled with
Helvetica + fonttype=42 (Illustrator-editable). The function below saves
the figure as PDF so you can edit it offline.

In [ ]:
from ecofoundation.reporting.plots import (
    niche_size_distribution, niches_per_group_bar, niche_centroids_spatial,
    co_occurrence_heatmap, niche_density_histogram, heterogeneity_histogram,
)
from ecofoundation.reporting.style import save_pdf

fig = niche_size_distribution(knn_niches)
save_pdf(fig, REPO / 'examples/notebooks/_out/01_niche_sizes.pdf')
fig

In [ ]:
fig = niches_per_group_bar(knn_niches)
fig

In [ ]:
fig = niche_centroids_spatial(
    adata, knn_niches,
    sample_key=data_cfg.sample_id_col,
    spatial_key=data_cfg.spatial_key,
    cell_sample=6000,
)
fig

In [ ]:
fig = co_occurrence_heatmap(stats)
fig

In [ ]:
fig = niche_density_histogram(stats)
fig

In [ ]:
fig = heterogeneity_histogram(stats)
fig

## 5. Saving the niche assignment

For downstream steps (graph construction, GNN training, unsupervised
clustering) we persist the long-form niche → cell membership table.

In [ ]:
long_rows = []
for nid in range(knn_niches.n_niches):
    for c in knn_niches.cells_per_niche[nid].tolist():
        long_rows.append({
            'niche_id': nid,
            'cell_index': c,
            'ego_cell': int(knn_niches.ego_cell[nid]),
            'patient': str(knn_niches.group_label[nid]),
        })
df = pd.DataFrame(long_rows)
df.head()